In [6]:
from pathlib import Path
import pandas as pd
from sqlalchemy import and_, func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

ENV = "PROD"  # per il risultato finale rilanciare con "PROD"
OUTPUT_DIR = Path("outputs/step0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()

SECTOR_NAME = "Quantum Computing"

# Classi CPC "core": prefisso senza spazi
SECTOR_CPC_PREFIXES = (
    "G06N10/",      # Quantum computing (dal 2019), include /20 /40 /60 /70 /80
    "G06N99/002",   # Quantum computers (collocazione storica, pre-2019)
    "Y10S977/933",  # Spintronics or quantum computing (collezione storica)
)

# Classi CPC "adiacenti": valide solo se accompagnate da testo quantistico
ADJACENT_CPC_PREFIXES = (
    "B82Y10/00",    # Nanotecnologie per elaborazione dell'informazione
    "H10N60/",      # Dispositivi superconduttori
    "H01L39/",      # Dispositivi superconduttori (vecchia collocazione)
    "H03K19/195",   # Logica superconduttiva
)

SECTOR_KEYWORDS = (
    "quantum comput",   # computer, computing, computation
    "qubit",
    "quantum bit",
    "quantum processor",
    "quantum gate",
    "quantum circuit",
    "quantum annealing",
    "quantum error correction",
    "quantum algorithm",
    "quantum simulator",
)

# Keyword generiche: usate SOLO in AND con le classi CPC adiacenti
ADJACENT_KEYWORDS = ("quantum", "josephson")
#La giunzione Josephson è il componente fisico su cui si basano i qubit superconduttori, che sono la piattaforma hardware più diffusa nel quantum computing
#Sono due elettrodi superconduttori separati da una barriera isolante sottilissima, tipicamente alluminio, spessa circa 1–2 nm.

START_YEAR = 1990   # anno di primo deposito della famiglia
END_YEAR = 2023

IPR_TYPE = "PI"  # "PI" = Patented Inventions (esclude i modelli di utilità)


def symbol_prefix_condition(symbol_column, codes):
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def keyword_condition(terms):
    title = func.lower(func.coalesce(TLS202_APPLN_TITLE.appln_title, ""))
    abstract = func.lower(func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""))
    return or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))


def sector_family_ids():
    family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")

    def base():
        return (
            db.query(family_id)
            .select_from(TLS201_APPLN)
            .filter(TLS201_APPLN.docdb_family_id > 0)  # esclude domande senza famiglia
            .filter(TLS201_APPLN.earliest_filing_year.between(START_YEAR, END_YEAR))
            .filter(TLS201_APPLN.ipr_type == IPR_TYPE) 
        )

    def join_cpc(q):
        return q.join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)

    def join_text(q):
        return (
            q.outerjoin(TLS202_APPLN_TITLE, T
                LS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
            .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        )

    # Perimetro CPC core
    by_cpc = join_cpc(base()).filter(
        symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_PREFIXES)
    )

    # Perimetro keyword specifiche
    by_keywords = join_text(base()).filter(keyword_condition(SECTOR_KEYWORDS))

    # CPC adiacente AND keyword generica
    by_adjacent = join_text(join_cpc(base())).filter(
        and_(
            symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, ADJACENT_CPC_PREFIXES),
            keyword_condition(ADJACENT_KEYWORDS),
        )
    )

    return by_cpc.union(by_keywords).union(by_adjacent)


sector_ids = sector_family_ids()

Aplln_id

In [7]:
# Funzione per estrarre tutti i singoli brevetti (appln_id) delle famiglie del settore
def sector_application_ids():
    """Query di tutti gli appln_id appartenenti alle famiglie del settore."""
    
    # 1. Otteniamo prima gli ID di famiglia tramite la tua query
    fam_ids_subquery = sector_family_ids().subquery().select()
    
    # 2. Selezioniamo tutte le applicazioni (tls201_appln) che appartengono a queste famiglie
    appln_query = (
        db.query(TLS201_APPLN.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(fam_ids_subquery))
        .filter(TLS201_APPLN.ipr_type == IPR_TYPE) # Esclude i modelli di utilità se configurato come "PI"
    )
    
    return appln_query

# Eseguiamo la query per ottenere la lista di tutte le applicazioni
sector_apps = sector_application_ids()

In [8]:
# 1. Calcoliamo il numero totale di famiglie uniche trovate
total_families = db.query(sector_ids.subquery()).count()

# 2. Calcoliamo il numero totale di singoli brevetti (applicazioni) trovati
total_applications = db.query(sector_apps.subquery()).count()

print(f"--- RISULTATI DELLA QUERY (Ambiente: {ENV}) ---")
print(f"Settore: {SECTOR_NAME}")
print(f"Totale Famiglie di brevetti trovate (invenzioni uniche): {total_families:,}")
print(f"Totale Applicazioni trovate (singoli depositi globali): {total_applications:,}")

--- RISULTATI DELLA QUERY (Ambiente: PROD) ---
Settore: Quantum Computing
Totale Famiglie di brevetti trovate (invenzioni uniche): 11,571
Totale Applicazioni trovate (singoli depositi globali): 26,075
